# 🧠 Notebook 03 — SQL Analysis

**Goal:** Load the clean data into a SQLite database and answer real analytical questions using pure SQL.

**What you will learn:**
- How to create a local SQLite database with Python
- How to load a DataFrame into a SQL table
- How to run SQL queries: SELECT, WHERE, GROUP BY, ORDER BY, JOIN
- How to combine SQL results with Pandas

**Input:** `data/processed/collections_clean.csv` + `data/processed/images_clean.csv`  
**Output:** `data/processed/neurovault.db` (SQLite database)

---
## 1. Import libraries

In [ ]:
import pandas as pd
import sqlite3
import os

print('Libraries imported ✅')

---
## 2. Load clean data

In [ ]:
df_c = pd.read_csv('data/processed/collections_clean.csv', low_memory=False)
df_i = pd.read_csv('data/processed/images_clean.csv', low_memory=False)

print(f'Collections: {df_c.shape[0]:,} rows × {df_c.shape[1]} columns')
print(f'Images:      {df_i.shape[0]:,} rows × {df_i.shape[1]} columns')

---
## 3. Create SQLite database

SQLite creates a local `.db` file — no server needed, no installation required.
We load both DataFrames as SQL tables inside this database.

In [ ]:
DB_PATH = 'data/processed/neurovault.db'

conn = sqlite3.connect(DB_PATH)

df_c.to_sql('collections', conn, if_exists='replace', index=False)
df_i.to_sql('images', conn, if_exists='replace', index=False)

print(f'Database created at: {DB_PATH} ✅')
print(f'  Table: collections → {df_c.shape[0]:,} rows')
print(f'  Table: images      → {df_i.shape[0]:,} rows')

# Helper function to run queries easily
def query(sql):
    return pd.read_sql_query(sql, conn)

---
## 4. Query 1 — How many studies per year?

**SQL concepts:** SELECT, GROUP BY, ORDER BY

In [ ]:
result = query("""
    SELECT 
        year,
        COUNT(*) AS total_studies
    FROM collections
    WHERE year IS NOT NULL
    GROUP BY year
    ORDER BY year ASC
""")

print('Studies per year:')
print(result.to_string(index=False))

---
## 5. Query 2 — What percentage of studies have a DOI?

**SQL concepts:** ROUND, AVG, calculated columns

In [ ]:
result = query("""
    SELECT
        COUNT(*) AS total_studies,
        SUM(has_doi) AS with_doi,
        SUM(1 - has_doi) AS without_doi,
        ROUND(AVG(has_doi) * 100, 1) AS pct_with_doi
    FROM collections
""")

print('DOI coverage:')
print(result.to_string(index=False))

---
## 6. Query 3 — Top 10 most prolific collections by number of images

**SQL concepts:** ORDER BY DESC, LIMIT

In [ ]:
result = query("""
    SELECT 
        id,
        name,
        number_of_images,
        year,
        has_doi
    FROM collections
    WHERE number_of_images IS NOT NULL
    ORDER BY number_of_images DESC
    LIMIT 10
""")

print('Top 10 collections by number of images:')
print(result.to_string(index=False))

---
## 7. Query 4 — Image modality breakdown

**SQL concepts:** GROUP BY, COUNT, percentage calculation

In [ ]:
result = query("""
    SELECT 
        modality,
        COUNT(*) AS total,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM images), 1) AS percentage
    FROM images
    GROUP BY modality
    ORDER BY total DESC
""")

print('Image modality breakdown:')
print(result.to_string(index=False))

---
## 8. Query 5 — Growth rate year over year

**SQL concepts:** Subquery, calculated columns, LAG simulation with self-join

In [ ]:
result = query("""
    WITH yearly AS (
        SELECT 
            year,
            COUNT(*) AS total
        FROM collections
        WHERE year IS NOT NULL AND year >= 2013
        GROUP BY year
    )
    SELECT 
        a.year,
        a.total,
        b.total AS prev_year_total,
        ROUND((a.total - b.total) * 100.0 / b.total, 1) AS growth_pct
    FROM yearly a
    LEFT JOIN yearly b ON a.year = b.year + 1
    ORDER BY a.year
""")

print('Year-over-year growth:')
print(result.to_string(index=False))

---
## 9. Query 6 — JOIN: images per collection (top 10)

**SQL concepts:** JOIN between two tables

In [ ]:
result = query("""
    SELECT 
        c.name AS collection_name,
        c.year,
        COUNT(i.id) AS image_count,
        i.modality
    FROM images i
    JOIN collections c ON CAST(i.collection AS TEXT) LIKE '%' || CAST(c.id AS TEXT) || '%'
    GROUP BY c.id, i.modality
    ORDER BY image_count DESC
    LIMIT 10
""")

print('Top collections by image count (with modality):')
print(result.to_string(index=False))

---
## 10. Close connection

In [ ]:
conn.close()
print('Database connection closed ✅')
print()
print('Next step → 04_visualization.ipynb 🚀')

---
## ✅ What we accomplished

- Created a local SQLite database from clean CSV data
- Answered 6 analytical questions using pure SQL
- Used SELECT, WHERE, GROUP BY, ORDER BY, LIMIT, JOIN, CTEs and subqueries
- Combined SQL results with Pandas for easy reading

**Next notebook:** `04_visualization.ipynb` — turn these findings into charts and visual stories.